# DNABERT2 local CPU full fine-tuning

This profile uses the same seed, predefined split policy, validation-MCC thresholding, and shared artifacts as the benchmark notebooks. It is intended for a local CPU machine. It is much slower than T4/A100/HPC training.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json

REPO_DIR = Path.cwd()
if not (REPO_DIR / "src" / "seqtrainer").exists():
    REPO_DIR = Path(r"C:\Users\Sgoff\MYfile\Desktop\PYThh\SeqTrainer")

CONFIG_PATH = REPO_DIR / "notebooks" / "final_training" / "config" / "dnabert2_local_cpu.toml"
OUTPUT_DIR = REPO_DIR / "outputs" / "benchmarks" / "dnabert2_local_cpu_ep_genomic_order"
assert CONFIG_PATH.exists(), CONFIG_PATH
print("Repository:", REPO_DIR)
print("Config:", CONFIG_PATH)
print("Python:", sys.executable)

In [ ]:
import pandas as pd
import tomli
import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
assert not torch.cuda.is_available(), "This notebook is the CPU profile; use the Colab/HPC notebook for CUDA."

clean_dir = REPO_DIR / "outputs" / "local_data" / "dnabert2_local_cpu_shared_split"
split_files = {
    "train": "train_EP_DNA_BERT2_genomic_order.csv",
    "validation": "eval_EP_DNA_BERT2_genomic_order.csv",
    "test": "test_EP_DNA_BERT2_genomic_order.csv",
}
frames = {name: pd.read_csv(clean_dir / filename) for name, filename in split_files.items()}
sets = {name: set(frame["sequence"].astype(str).str.upper().str.replace("U", "T", regex=False)) for name, frame in frames.items()}
for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    assert not sets[left].intersection(sets[right]), f"Cross-split leakage: {left}/{right}"
print({name: len(frame) for name, frame in frames.items()})

In [ ]:
manifest_command = [
    sys.executable, "-m", "seqtrainer.cli.main", "benchmark", "manifest",
    str(CONFIG_PATH), "--base-dir", str(REPO_DIR), "--output-dir", str(OUTPUT_DIR),
]
subprocess.run(manifest_command, cwd=REPO_DIR, check=True)

## Full local run

This cell can take many hours on CPU. It uses batch size 1, sequence length 70, float32, four epochs maximum, gradient accumulation 16, and early stopping on validation MCC.

In [ ]:
run_env = os.environ.copy()
run_env["HF_HUB_OFFLINE"] = "1"
run_env["TOKENIZERS_PARALLELISM"] = "false"

run_command = [
    sys.executable, "-u", "-m", "seqtrainer.cli.main", "benchmark", "run",
    str(CONFIG_PATH), "--base-dir", str(REPO_DIR), "--output-dir", str(OUTPUT_DIR), "--strict",
]
print("Running:", " ".join(run_command))
subprocess.run(run_command, cwd=REPO_DIR, env=run_env, check=True)

In [ ]:
metrics_path = OUTPUT_DIR / "metrics.csv"
assert metrics_path.exists(), metrics_path
metrics = pd.read_csv(metrics_path)
display(metrics)
print("Artifacts:", [path.name for path in sorted(OUTPUT_DIR.iterdir())])